# Method for achieving velocity mached interaction between robots and object

## Preliminaries

### Bezier curves
To parametrise the trajectories in a way that is scalable with MILP we use bezier curves. Robots have order 5 with a total of 6 control points (CP) and objects have order 1 with 2 control points. Robots have a higher order in order for the solver to change its speed, ie for it to be able to move on its own. Objects are limited to first order so that their speed cannot be changed by themselves, the solver has to use the interactions between robots and objects to make it move and achieve the desired transport or whatever other constraints it has.

Furthermore, the trajectory of the robots and objects are split up into a temporal curve representing time, and a spatial curve representing position.

## Moving the objects

### High level interactions scheme

The interactions between robots and objects follows the following scheme:
* Robot matches velocity of object 
* Robot imparts a new velocity on the object
* Interaction stops (disengagement)

There is an important caveat here, the robot cannot move the object in any way, since they are not mechanically linked and the robot can only apply force through a single contact point, we are limited in how the robot can effect the objects velocity.

## Continuity and Bezier curves compatibility

The robot and object trajectories are modeled with Bézier curves, which may have different orders. Because of this mismatch, it is not possible to enforce continuity of time, position, and velocity at all control points (CPs) during an interaction. To address this, we relax certain continuity constraints while making the following assumptions.  

---

### Notation
- Each interaction curve is denoted $ C^{\text{int}} $. The preceding and following curves are $ C^{\text{pre}} $ and $ C^{\text{post}} $.  
- Control points are labeled $ \text{cp}_1, \text{cp}_2, \dots, \text{cp}_n $. For example, $ C^{\text{pre}}(\text{cp}_1) $ denotes the first CP of the preceding curve.  
- Robot state: $ (t_r, x_r, v_r) $ for time, position, velocity.  
- Object state: $ (t_o, x_o, v_o) $.  

---

### Continuity and Interaction Constraints

1. **Robot trajectory continuity**  

   The robot trajectory is continuous across all consecutive curves. For curve index $ z $ and its successor $ z+1 $:  

   $$
   \begin{aligned}
   t_r^{(z)}(\text{cp}_n) &= t_r^{(z+1)}(\text{cp}_1), \\
   x_r^{(z)}(\text{cp}_n) &= x_r^{(z+1)}(\text{cp}_1), \\
   v_r^{(z)}(\text{cp}_n) &= v_r^{(z+1)}(\text{cp}_1).
   \end{aligned}
   $$

   This ensures the robot trajectory is physically consistent with no discontinuities.

2. **Interaction start condition**  

   At the beginning of an interaction curve, the robot and object states coincide:  

   $$
   \big(t_r^{\text{int}}(\text{cp}_1), x_r^{\text{int}}(\text{cp}_1), v_r^{\text{int}}(\text{cp}_1)\big) 
   =
   \big(t_o^{\text{int}}(\text{cp}_1), x_o^{\text{int}}(\text{cp}_1), v_o^{\text{int}}(\text{cp}_1)\big).
   $$

   Since the object has only one velocity CP and two position CPs per curve, it cannot be forced to exactly follow the robot’s path during the interaction.

3. **Interaction assumption**  

   During the interaction, the robot and object trajectories are assumed identical (conceptually, not strictly enforced by the solver):  

   $$
   (t_r^{\text{int}}(s), x_r^{\text{int}}(s), v_r^{\text{int}}(s)) 
   =
   (t_o^{\text{int}}(s), x_o^{\text{int}}(s), v_o^{\text{int}}(s)), 
   \quad \forall s \in [0,1].
   $$

   Since the object’s velocity is defined by only a single control point per curve, the solver treats it as constant during $C^{\text{int}} $. In reality, the robot influences the object’s velocity continuously, but this update is deferred until the next curve.

4. **Post-interaction condition**  

   At the beginning of the curve after the interaction, the object is aligned with the robot:  

   $$
   \big(t_o^{\text{post}}(\text{cp}_1), x_o^{\text{post}}(\text{cp}_1), v_o^{\text{post}}(\text{cp}_1)\big) 
   =
   \big(t_r^{\text{post}}(\text{cp}_1), x_r^{\text{post}}(\text{cp}_1), v_r^{\text{post}}(\text{cp}_1)\big).
   $$

   To achieve this, we permit discontinuities in the object’s state between the end of the interaction and the start of the following curve:  

   $$
   (x_o^{\text{int}}(\text{cp}_n), v_o^{\text{int}}(\text{cp}_n)) 
   \neq 
   (x_o^{\text{post}}(\text{cp}_1), v_o^{\text{post}}(\text{cp}_1)).
   $$


## Constraints on Movement During Interactions

During an interaction, the robot and object are not mechanically linked but instead rely on a single contact point for force transfer. Because of this, the robot’s movement must obey strict constraints.  

- If the robot were to change the direction of applied force within a curve, it would require changing the contact point on the object to maintain pushing consistency.  
- Since the robot cannot independently control the force components in multiple dimensions, diagonal forces must instead be synthesized by varying the interaction angle relative to the object’s current and desired velocity.  

To simplify these conditions, the following constraints are imposed on the robot’s movement during an interaction:

---

### Constraints

1. **Single-dimension influence**  
   The robot may only influence the object’s velocity along **one spatial dimension** $(x \text{ or } y)$ per Bézier curve.  

2. **Monotonic force sign**  
   Within a single curve, the applied force must maintain a **constant sign** (purely positive or purely negative).  

---

### Formalization

Let $ v_x^k $ and $ v_y^k $ denote the object’s velocities in the $x$- and $y$-dimensions at discrete time $t_k$. For each interaction, the robot must choose one of four possible force applications:

- **Positive \(y\)-direction**:  
  $$
  v_y^{k+1} - v_y^k \geq 0
  $$
  Force is applied to increase positive \(y\)-velocity or decrease negative \(y\)-velocity.  

- **Negative \(y\)-direction**:  
  $$
  v_y^{k+1} - v_y^k \leq 0
  $$
  Force is applied to increase negative \(y\)-velocity or decrease positive \(y\)-velocity.  

- **Positive \(x\)-direction**:  
  $$
  v_x^{k+1} - v_x^k \geq 0
  $$
  Force is applied to increase positive \(x\)-velocity or decrease negative \(x\)-velocity.  

- **Negative \(x\)-direction**:  
  $$
  v_x^{k+1} - v_x^k \leq 0
  $$
  Force is applied to increase negative \(x\)-velocity or decrease positive \(x\)-velocity.  

Thus, the robot is restricted to applying forces aligned with the **four cardinal directions**. While diagonal forces cannot be produced within a single Bézier curve, diagonal motions can still emerge as a composition of successive interactions along $x$ and $y$.  

---
### Parametric Consideration

The velocity of the robot along the Bezier curve is naturally expressed as

$$
v = \frac{dr}{dh}
$$

where  
- $dr$ is the derivative of the spatial curve (position derivative),  
- $dh$ is the derivative of the temporal curve (time derivative).  

Enforcing constraints on $v$ directly is difficult in a MILP framework, because it introduces **division of decision variables**. For example, a condition such as

$$
v_2 - v_1 \geq 0 \quad \Rightarrow \quad \frac{dr_2}{dh_2} - \frac{dr_1}{dh_1} \geq 0
$$

is nonlinear or at best bilinear and cannot be represented using linear constraints. Handling this would require McCormick envelopes or nonconvex quadratic formulations, both of which increase complexity and degrade solver performance.

---


If we instead enforce that $dh$ is constant across all control points of a given interaction, i.e.

$$
dh_{cp} = H \quad \text{for all cp, with } H > 0,
$$

then the velocity becomes

$$
v_{cp} = \frac{dr_{cp}}{H}.
$$

Since $H$ is constant and strictly positive, comparing velocities is equivalent to comparing the corresponding $dr$ values:

$$
v_2 - v_1 = \frac{dr_2}{H} - \frac{dr_1}{H} = \frac{dr_2 - dr_1}{H}.
$$

Because dividing by a positive constant preserves inequality direction, we obtain the equivalence

$$
v_2 - v_1 \geq 0 \quad \Leftrightarrow \quad dr_2 - dr_1 \geq 0.
$$

This means that all velocity monotonicity constraints (such as push or brake conditions) can be imposed **directly on $dr$** without involving $dh$, making the formulation purely linear.

---


- **Advantage:** Constraints remain linear and easy to enforce.  
- **Limitation:** Timing flexibility is lost, since $dh$ is no longer a decision variable but a fixed constant across the interaction.  



## Constraints on Curves
### Robot interactions per bezier curve
To constrain the robot from interacting with several different objects, we constrain each robots bezier curve to at most be able to interact with one other bezier curve, this can be from any object but each robot bezier curve can at most interact with another bezier curve. 

### Object interactions
In order for the interactions of different robots to not happen on consecutive object bezier curves, we constrain the object to always require a free/uninteracted curve after each interaction.

### Robot Consecutive interactions
In order for one robot to not both accelerate and decelerate an object in different curves, we force the object to change which robot it interacts with next after each interactions. For eg: If robot1 pushes the object, before robot1 can ever interact with the object again, robot2  has to interact with it.
This both constrains it from pushing and catching the object itself, and from catching the object and then pushing it.

But an important caveat is that this constraint also makes the robot unable to catch it in one dimension, bring it to a stop, and then push it in another direction, for instance when the object wants to make a 90 degree bend, the catching and pushing have to be done by different robots.

To this extent, I have also made a constrain that allows this sort of behabiour but does not allow consecutive interactions on the same dimension.
An example: Robot1 pushes the object in y direction, object has a free curve, robot1 pushes it in x direction, object has free curve ...

This however creates a lot of branching for the solver and is time consuming for large planning horizons. I am undecided on which ot use for now, but im continuing with not allowing consecutive interactions form one robot since this is the least challenging for the replanner.